# Airbnb Price Prediction — Malaga

Project notebook. **Done so far:** load data + cleaning.


In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CITY, TARGET_COLUMN
from src.data import load_calendar, load_listings, load_reviews, save_modeling_table

print(f"City: {CITY}")

City: Malaga


## 1. Load raw data

In [2]:
listings = load_listings()
reviews = load_reviews()
calendar = load_calendar()

print(f"Listings: {listings.shape}")
print(f"Reviews:  {reviews.shape}")
if calendar is not None:
    print(f"Calendar: {calendar.shape}")
else:
    print("Calendar: not downloaded yet (optional)")

listings.head()

Listings: (9714, 79)
Reviews:  (487221, 2)
Calendar: not downloaded yet (optional)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,96033,https://www.airbnb.com/rooms/96033,20250930030808,2025-09-30,city scrape,"Bonito piso a 200m de la playa, El Palo (Málaga)",Do you have a backpacker spirit and are lookin...,"200 metres from the beaches of El Palo, Malaga...",https://a0.muscache.com/pictures/hosting/Hosti...,510467,...,4.93,4.44,4.61,ESFCTU0000290200003588210000000000000000VUT/MA...,f,1,1,0,0,1.88
1,166473,https://www.airbnb.com/rooms/166473,20250930030808,2025-09-30,city scrape,Perfect Location In Malaga,This apartment is rented out by the room - new...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,793360,...,4.91,4.80,4.72,NaN,f,5,1,4,0,0.59
2,330760,https://www.airbnb.com/rooms/330760,20250930030808,2025-09-30,city scrape,Malaga Lodge Guesthouse Double room-shared bath.,The Lodge is set in a charming town house in L...,Málaga Lodge is situated next to the famous Sa...,https://a0.muscache.com/pictures/85419390/38a9...,1687526,...,4.62,4.52,4.48,ESHFTU0000290200004234200060000000000000VFT/MA...,t,6,4,2,0,0.41
3,340024,https://www.airbnb.com/rooms/340024,20250930030808,2025-09-30,city scrape,NEW APARTMENT IN MALAGA CENTER,Welcome to Málaga!<br />This is a modern and e...,It is a central area and has all kinds of serv...,https://a0.muscache.com/pictures/hosting/Hosti...,1725690,...,4.79,4.72,4.77,VFT/MA/02334,f,1,1,0,0,2.11
4,358541,https://www.airbnb.com/rooms/358541,20250930030808,2025-09-30,city scrape,Casa La Maga - Apartment for happy people,"For years, Raúl and I were super happy in this...",The apartment is in the very heart of Malaga C...,https://a0.muscache.com/pictures/miso/Hosting-...,1526932,...,4.97,4.80,4.78,VFT/MA/02288,f,1,1,0,0,2.48


## 2. Exploratory analysis

TODO: price distribution, missing values, spatial map, plots for presentation.

## 3. Cleaning & preprocessing

What the cleaning step does:
- Converts types (`price` → number, `t`/`f` → 0/1, dates → datetime)
- Drops useless columns (100% empty like `neighbourhood_group_cleansed`, URLs, leaky features)
- Removes price outliers and listings outside Malaga coordinates
- Aggregates reviews per listing and adds `has_reviews` flag

**Missing values that are normal:**
- `neighborhood_overview`, `host_about` — hosts often leave these blank
- `review_scores_*`, `first_review`, `last_review` — missing on ~10% of listings with **no reviews yet** (same rows)

In [3]:
from src.data import cleaning_report

output_path = save_modeling_table()
df = pd.read_csv(output_path)

print(f"Saved {df.shape} -> {output_path}")
print(f"\nPrice summary:")
display(df[TARGET_COLUMN].describe())

print("\nColumn types and missing values (sorted by missing %):")
display(cleaning_report(df))

Saved (8677, 68) -> C:\Users\RPC\Documents\ML_Final_projet\data\processed\malaga_modeling_table.csv

Price summary:


count    8677.000000
mean      211.850870
std       748.454081
min        26.000000
25%        77.000000
50%       102.000000
75%       146.000000
max      9000.000000
Name: price, dtype: float64


Column types and missing values (sorted by missing %):


,column,dtype,missing_pct,unique
3,neighborhood_overview,object,56.7,2954
8,host_about,object,42.5,1389
7,host_location,object,23.1,189
47,review_scores_rating,float64,9.9,152
52,review_scores_location,float64,9.9,136
...,...,...,...,...
56,calculated_host_listings_count,int64,0.0,57
55,instant_bookable,int64,0.0,2
62,amenities_count,int64,0.0,86
63,review_count,int64,0.0,445


In [4]:
# Check: no fully empty columns left
empty_cols = [c for c in df.columns if df[c].isna().all()]
print("Fully empty columns:", empty_cols if empty_cols else "none (good)")

# Missing review scores = listings without reviews (expected pattern)
no_reviews = df[df["has_reviews"] == 0]
print(f"\nListings without reviews: {len(no_reviews)} ({len(no_reviews)/len(df):.1%})")
print("These rows miss review_scores_*, first_review, last_review — that is normal.")

df.dtypes.value_counts()

Fully empty columns: none (good)

Listings without reviews: 856 (9.9%)
These rows miss review_scores_*, first_review, last_review — that is normal.


int64      25
float64    25
object     18
Name: count, dtype: int64

## 4. Baseline models (tabular + spatial)

TODO: train/test split, Linear Regression, Random Forest, evaluate MAE / RMSE / R².

## 5. Hybrid model (text + tabular)

TODO: TF-IDF on descriptions/reviews combined with tabular features.

## 6. Model comparison

TODO: compare all models and save plots to `reports/figures/`.